**NLP?**
Is a field that combines linguistics and AI to enable computers to process and understand human language.

**SECTION 1: NLP Flow / BluePrint**

1. **Import the Libraries/Modules**
* Purpose: Import the important libraries that will help to accomplish the goal
* Key term:
    * Module: a library of important code that you can reuse in other programs.

2. **Data Collection, Cleaning, and Exploraion**
* Purpose: Get the data (e.g., SMS messages).
* Key term:
    * Dataset: A collection of data.

3. **Text Cleaning (Normalization)**
* Purpose: Make text consistent and remove unnecessary variation.
* What we do:
    * Convert all text to Lowercase text ("Hello" → "hello")
    * Remove punctuation
* Key terms:
    * Normalization: Making text uniform so the model doesn’t treat similar things differently.

4. **Tokenization**
* Purpose: Break text into smaller pieces (words).
    * Example:
        "I love AI" → ["I", "love", "AI"]
* Key term: 
    * Token: A single unit (usually a word).

5. **Stop Words Removal**
* Purpose: Remove common words that don’t add much meaning.
    * Example: "the", "is", "and"
* Key term:
    * Stop words: Frequently used words that carry little useful information.

6. **Stemming / Lemmatization** 
* Purpose: Reduce words to their base form.
    * Example:
        * Stemming: "running" → "run"
        * Lemmatization: "better" → "good"
* Key terms:
    * Stem: Crude root form of a word
    * Lemma: Proper dictionary form of a word

7. **Vectorization**
* Purpose: Convert text into numbers so a model can understand it.
    * Example:
        "love AI" → [0,1,1,0]
* Key terms:
    * Vector: A list of numbers
    * Bag of Words: Count of words, ignoring order
    * Feature: A word used as input to the model

8. **Model Training**
* Purpose: Teach a machine learning model to recognize patterns.
    * Example: Learn what spam messages look like.
* Key terms:
    * Model: A function that learns from data
    * Training: The process of learning patterns

9. **Prediction / Evaluation**
* Purpose: Test how well the model performs.
    * Example: Predict if a new SMS is spam or ham.
* Key terms:
    * Prediction: Model output
    * Accuracy: How often the model is correct

### 1. Import the Libraries / Modules, that will help achieve the desired goal

In [1]:
# import the base libraries for loading data and visualisation
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#view the graphs inline
%matplotlib inline              

# import the NLTK library and useful functions
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

# import the sklearn functions useful for NLP
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score



#### 2. Data Collection And Exploration

In [3]:
# Load the data
sms_data = pd.read_csv('../datasets/spam.csv', encoding='latin-1')

In [4]:
# View the first 5 columns
sms_data.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


**Observations**
The data has the label column named as v1 and the features column as v2. The rest of the columns will not contribute mutch to our goal as they are unnamed and seem to contain missing values (more on this later)

Lets explore the data next (Note: in the future, use the info() function to get the data info all at once))


In [9]:
# View the shape of the data -> total rows and columns i.e, (rows, columns)
print("The shape of the data is: ", sms_data.shape, "\n", 
      "The total missing values per column is: ", sms_data.isna().sum(), "\n",
       "The data types for each column","\n", 
       sms_data.dtypes, "\n",
        "Finally, the summary of the data is as follows:", "\n",
         sms_data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   v1          5572 non-null   object
 1   v2          5572 non-null   object
 2   Unnamed: 2  50 non-null     object
 3   Unnamed: 3  12 non-null     object
 4   Unnamed: 4  6 non-null      object
dtypes: object(5)
memory usage: 217.8+ KB
The shape of the data is:  (5572, 5) 
 The total missing values per column is:  v1               0
v2               0
Unnamed: 2    5522
Unnamed: 3    5560
Unnamed: 4    5566
dtype: int64 
 The data types for each column 
 v1            object
v2            object
Unnamed: 2    object
Unnamed: 3    object
Unnamed: 4    object
dtype: object 
 Finally, the summary of the data is as follows: 
 None


#

In the future, use the info() function to get the summary as quickly as possible

* The Unnamed columns have many missing values and will not contribute to the goal of this project and will be removed. V2 and v1 columns have no missing values and will be kept.

In [13]:
# remove the unneccesary columns and rename the remaining columns to something meaningful
sms_data = sms_data[['v1', 'v2']]
sms_data.columns = ['label', 'message']
sms_data.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


Since our goal is to classify the messages, lets convert the labels: ham = 0 and spam = 1 

In [14]:
# convert the label column to binary values
sms_data['label'] = sms_data['label'].map({'ham': 0, 'spam': 1})
# verify
sms_data.head()

C:\Users\Isaac Maake\AppData\Local\Temp\ipykernel_17700\2266904210.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sms_data['label'] = sms_data['label'].map({'ham': 0, 'spam': 1})


,label,message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
#